# Generalization-first Indian currency classifier

This notebook avoids optimistic random-image accuracy. It performs dataset audits, duplicate/perceptual-near-duplicate checks, grouped splits when source groups can be inferred, a strictly untouched test set, hard augmentations, class-balanced training, calibration, and external-image evaluation.

**Important:** random train/test splitting can leak near-identical photos. A high score is not evidence of real-world accuracy unless the test set contains different notes, backgrounds, cameras, and lighting.


In [ ]:
import os, re, json, random, hashlib, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Configuration and dataset audit

Set `DATASET_ROOT` to the folder containing the denomination folders. The audit reports invalid files, dimensions, aspect ratios, class balance, exact duplicates, and perceptual duplicates. Remove duplicates before splitting.


In [ ]:
DATASET_ROOT = Path("Indian Currency Dataset")
OUTPUT_DIR = Path("currency_models_generalized")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = ["10", "20", "50", "100", "200", "500", "2000"]
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
if not DATASET_ROOT.exists(): raise FileNotFoundError(DATASET_ROOT)

records=[]; bad=[]
for cls in CLASS_NAMES:
    folder=DATASET_ROOT/cls
    if not folder.exists(): print("Missing class folder:", folder); continue
    for path in folder.rglob("*"):
        if path.suffix.lower() not in EXTS: continue
        try:
            with Image.open(path) as im:
                im.verify()
            with Image.open(path) as im:
                w,h=im.size
            sha=hashlib.sha256(path.read_bytes()).hexdigest()
            records.append({"path":str(path),"class_name":cls,"width":w,"height":h,"aspect":w/max(h,1),"sha256":sha})
        except Exception as e: bad.append((str(path),str(e)))
df=pd.DataFrame(records)
if df.empty: raise RuntimeError("No readable images found")
print("Images:",len(df),"\nClasses:",df.class_name.value_counts().sort_index().to_dict(),"\nBad files:",len(bad))
print(df.groupby("class_name")[["width","height","aspect"]].agg(["count","min","median","max"]))
print("Exact duplicate files:", int(df.sha256.duplicated(keep=False).sum()))


In [ ]:
# Remove exact duplicate files while retaining one copy per content hash.
# For near-duplicates, use pHash if imagehash is installed; otherwise the audit below is still valid.
keep = ~df.duplicated("sha256", keep="first")
df_clean = df.loc[keep].reset_index(drop=True)
try:
    import imagehash
    ph=[]
    for p in df_clean.path:
        ph.append(str(imagehash.phash(Image.open(p).convert("RGB"))))
    df_clean["phash"]=ph
    print("Perceptual hash collisions:", int(df_clean.phash.duplicated(keep=False).sum()))
except ImportError:
    print("Optional: pip install ImageHash to audit near-duplicate images")
print("Usable unique images:",len(df_clean))


## 2. Leakage-resistant split

If filenames contain a capture/session identifier, populate `group_id` so photos from the same recording session stay in one partition. If no reliable group is available, the notebook uses a stratified split but clearly labels it as weaker. For the strongest estimate, add a separate `external_test/` folder collected with a different phone/session.


In [ ]:
def infer_group(path):
    # Customize this for your filenames. Examples: note01_sessionA_003.jpg -> note01_sessionA
    stem=Path(path).stem.lower()
    stem=re.sub(r"([_-]?(img|image|photo|frame)?[_-]?\d+)$", "", stem)
    return stem if stem else Path(path).parent.name

df_clean["group_id"]=df_clean.path.map(infer_group)
idx=np.arange(len(df_clean)); y=df_clean.class_name.to_numpy(); groups=df_clean.group_id.to_numpy()
use_groups = df_clean.group_id.nunique() < len(df_clean) * 0.95 and df_clean.groupby("group_id").class_name.nunique().max() == 1
if use_groups:
    gss=GroupShuffleSplit(n_splits=1,test_size=.30,random_state=SEED)
    train_idx,temp_idx=next(gss.split(idx,y,groups))
    gss2=GroupShuffleSplit(n_splits=1,test_size=.50,random_state=SEED)
    val_rel,test_rel=next(gss2.split(temp_idx,y[temp_idx],groups[temp_idx]))
    val_idx,temp_test= temp_idx[val_rel], temp_idx[test_rel]
    test_idx=temp_test
    print("Used grouped split")
else:
    train_idx,temp_idx=train_test_split(idx,test_size=.30,stratify=y,random_state=SEED)
    val_idx,test_idx=train_test_split(temp_idx,test_size=.50,stratify=y[temp_idx],random_state=SEED)
    print("Used stratified split; collect session/device groups for a stronger test")

splits={"train":train_idx.tolist(),"val":val_idx.tolist(),"test":test_idx.tolist()}
with open(OUTPUT_DIR/"split_manifest.json","w") as f: json.dump({"records":df_clean.to_dict("records"),"splits":splits,"classes":CLASS_NAMES},f,indent=2)
for name, ids in splits.items(): print(name, len(ids), df_clean.iloc[ids].class_name.value_counts().sort_index().to_dict())


## 3. Real-world transforms

Currency notes should not be horizontally flipped: text, portraits, and security features are directional. The training pipeline simulates camera distance, perspective, shadows, glare, blur, compression, and uneven lighting. Validation/test use only deterministic preprocessing.


In [ ]:
IMG_SIZE=224
MEAN=[.485,.456,.406]; STD=[.229,.224,.225]
train_tf=transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(IMG_SIZE,scale=(.65,1.0),ratio=(.70,1.35)),
    transforms.RandomPerspective(distortion_scale=.25,p=.45),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0,translate=(.08,.08),scale=(.85,1.10),shear=8),
    transforms.ColorJitter(brightness=.35,contrast=.35,saturation=.30,hue=.04),
    transforms.RandomApply([transforms.GaussianBlur(5,(.1,2.0))],p=.25),
    transforms.RandomGrayscale(p=.04), transforms.ToTensor(),
    transforms.RandomErasing(p=.20,scale=(.01,.10),ratio=(.3,3.3),value="random"),
    transforms.Normalize(MEAN,STD)])
eval_tf=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

class CurrencyDataset(Dataset):
    def __init__(self, frame, indices, tf): self.frame=frame.iloc[indices].reset_index(drop=True); self.tf=tf; self.label={c:i for i,c in enumerate(CLASS_NAMES)}
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        with Image.open(row.path) as im: im=im.convert("RGB")
        return self.tf(im), self.label[row.class_name], row.path
train_ds=CurrencyDataset(df_clean,train_idx,train_tf); val_ds=CurrencyDataset(df_clean,val_idx,eval_tf); test_ds=CurrencyDataset(df_clean,test_idx,eval_tf)
counts=Counter(df_clean.iloc[train_idx].class_name); weights=np.array([1.0/counts[c] for c in df_clean.iloc[train_idx].class_name])
sampler=WeightedRandomSampler(torch.DoubleTensor(weights),len(weights),replacement=True)
workers=2 if os.name!="nt" else 0
train_loader=DataLoader(train_ds,batch_size=32,sampler=sampler,num_workers=workers,pin_memory=torch.cuda.is_available())
val_loader=DataLoader(val_ds,batch_size=64,shuffle=False,num_workers=workers,pin_memory=torch.cuda.is_available())
test_loader=DataLoader(test_ds,batch_size=64,shuffle=False,num_workers=workers,pin_memory=torch.cuda.is_available())


## 4. Model and training

The backbone starts from ImageNet features. The classifier uses moderate dropout and label smoothing. Early stopping is based on validation macro-F1 rather than accuracy, so a majority class cannot hide weak denominations.


In [ ]:
def build_model():
    try: weights=models.MobileNet_V3_Large_Weights.DEFAULT
    except Exception: weights=None
    m=models.mobilenet_v3_large(weights=weights)
    for p in m.features.parameters(): p.requires_grad=True
    n=m.classifier[0].in_features
    m.classifier=nn.Sequential(nn.Linear(n,512),nn.Hardswish(),nn.Dropout(.35),nn.Linear(512,128),nn.Hardswish(),nn.Dropout(.20),nn.Linear(128,len(CLASS_NAMES)))
    return m
model=build_model().to(device)
criterion=nn.CrossEntropyLoss(label_smoothing=.10)
optimizer=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=30,eta_min=1e-6)
scaler=torch.amp.GradScaler("cuda",enabled=device.type=="cuda")

def run_epoch(loader,training=False):
    model.train(training); total_loss=0; yt=[]; yp=[]
    for x,y,_ in loader:
        x,y=x.to(device),y.to(device)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda",enabled=device.type=="cuda"):
            out=model(x); loss=criterion(out,y)
        if training:
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),2.0); scaler.step(optimizer); scaler.update()
        total_loss+=loss.item()*len(y); yt.extend(y.detach().cpu().numpy()); yp.extend(out.argmax(1).detach().cpu().numpy())
    return total_loss/len(loader.dataset), np.array(yt), np.array(yp)

best=-1; patience=7; stale=0; EPOCHS=30
for epoch in range(1,EPOCHS+1):
    tr_loss,_,_=run_epoch(train_loader,True); va_loss,yt,yp=run_epoch(val_loader,False)
    macro=f1_score(yt,yp,average="macro",zero_division=0); bal=balanced_accuracy_score(yt,yp)
    if macro>best:
        best=macro; stale=0
        torch.save({"model_state_dict":model.state_dict(),"class_names":CLASS_NAMES,"mean":MEAN,"std":STD,"img_size":IMG_SIZE,"val_macro_f1":float(macro)},OUTPUT_DIR/"best_currency_model.pth")
    else: stale+=1
    scheduler.step()
    print(f"{epoch:02d} train_loss={tr_loss:.4f} val_loss={va_loss:.4f} val_macro_f1={macro:.4f} val_bal_acc={bal:.4f}")
    if stale>=patience: print("Early stopping"); break


## 5. Honest evaluation

Report accuracy, balanced accuracy, macro-F1, per-class recall, and the confusion matrix. Also evaluate a separate external folder if available.


In [ ]:
ckpt=torch.load(OUTPUT_DIR/"best_currency_model.pth",map_location=device); model.load_state_dict(ckpt["model_state_dict"]); model.eval()
loss,yt,yp=run_epoch(test_loader,False)
print("Held-out accuracy:",float((yt==yp).mean()))
print("Held-out balanced accuracy:",balanced_accuracy_score(yt,yp))
print(classification_report(yt,yp,target_names=CLASS_NAMES,digits=4,zero_division=0))
print(pd.DataFrame(confusion_matrix(yt,yp),index=CLASS_NAMES,columns=CLASS_NAMES))


In [ ]:
# Optional external evaluation: create external_test/10, external_test/20, ... with photos from a different device/session.
EXTERNAL_ROOT=Path("external_test")
if EXTERNAL_ROOT.exists():
    ext_rows=[]
    for cls in CLASS_NAMES:
        for p in (EXTERNAL_ROOT/cls).rglob("*") if (EXTERNAL_ROOT/cls).exists() else []:
            if p.suffix.lower() in EXTS: ext_rows.append({"path":str(p),"class_name":cls})
    if ext_rows:
        ext_df=pd.DataFrame(ext_rows); ext_idx=np.arange(len(ext_df))
        class External(Dataset):
            def __len__(self): return len(ext_df)
            def __getitem__(self,i):
                with Image.open(ext_df.iloc[i].path) as im: im=im.convert("RGB")
                return eval_tf(im), CLASS_NAMES.index(ext_df.iloc[i].class_name), ext_df.iloc[i].path
        ext_loader=DataLoader(External(),batch_size=64,shuffle=False)
        _,ey,ep=run_epoch(ext_loader,False)
        print("EXTERNAL accuracy:",float((ey==ep).mean()),"macro-F1:",f1_score(ey,ep,average="macro",zero_division=0))
    else: print("external_test exists but contains no readable class images")
else: print("Create external_test/<denomination>/ with new phone/session images for the real generalization score")


## 6. Export for deployment

The exported ONNX model uses the same preprocessing and canonical class order. Copy `currency_model.onnx` and `labels.json` to the Hugging Face Space.


In [ ]:
import shutil
model.eval(); onnx_path=OUTPUT_DIR/"currency_model.onnx"
dummy=torch.randn(1,3,IMG_SIZE,IMG_SIZE,device=device)
torch.onnx.export(model,dummy,str(onnx_path),input_names=["input_image"],output_names=["class_logits"],dynamic_axes={"input_image":{0:"batch"},"class_logits":{0:"batch"}},opset_version=17)
shutil.copy2(onnx_path,"currency_model.onnx")
with open(OUTPUT_DIR/"labels.json","w") as f: json.dump({"class_names":CLASS_NAMES,"mean":MEAN,"std":STD,"img_size":IMG_SIZE},f,indent=2)
shutil.copy2(OUTPUT_DIR/"labels.json","labels.json")
print("Saved",onnx_path,"and currency_model.onnx")
